# Data Loading and Embeddings

This notebook teaches you how to load documents into Neo4j and enable semantic search with vector embeddings using Databricks Model Serving.

**Learning Objectives:**
- Create the Document-Chunk graph structure with `NEXT_CHUNK` relationships
- Generate embeddings using Databricks Foundation Model APIs (GTE)
- Create a vector index for similarity search

---

## The Document-Chunk Pattern

We split documents into smaller **chunks** because LLMs have context limits and smaller chunks enable precise retrieval. The graph structure preserves document order and provenance:

```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
```

**Embeddings** are vectors that capture semantic meaning—similar texts produce similar vectors, enabling "meaning-based" search rather than keyword matching.

## Setup

In [ ]:
# Install dependencies for Databricks Model Serving
%pip install neo4j-graphrag databricks-langchain python-dotenv pydantic-settings nest-asyncio -q

In [ ]:
from neo4j_graphrag.indexes import create_vector_index
from data_utils import Neo4jConnection, DataLoader, split_text, get_embedder

## Load Sample Data

Load text from `company_data.txt` representing content from an SEC 10-K filing.

In [ ]:
loader = DataLoader("company_data.txt")
SAMPLE_TEXT = loader.text
DOCUMENT_PATH = "form10k-sample/apple-2023-10k.pdf"

metadata = loader.get_metadata()
print(f"Loaded from: {metadata['name']}")
print(f"Sample text length: {metadata['size']} characters")
print(f"\n{SAMPLE_TEXT}")

## Connect to Neo4j

In [ ]:
neo4j = Neo4jConnection().verify()
driver = neo4j.driver
neo4j.clear_graph()  # Clean start

## Split Text into Chunks

Use `FixedSizeSplitter` to split text into chunks with configurable size and overlap.

In [ ]:
chunks_text = split_text(SAMPLE_TEXT, chunk_size=400, chunk_overlap=50)

print(f"Split into {len(chunks_text)} chunks:\n")
for i, chunk in enumerate(chunks_text):
    print(f"Chunk {i}: {len(chunk)} chars")
    print(f"  {chunk}\n")

## Generate Embeddings

Generate embedding vectors for each chunk using Databricks Foundation Model APIs (GTE - General Text Embedding).

In [ ]:
embedder = get_embedder()
print(f"Embedder initialized: {embedder.endpoint}")

In [ ]:
chunk_embeddings = []
for i, text in enumerate(chunks_text):
    embedding = embedder.embed_query(text)
    chunk_embeddings.append({
        "text": text,
        "index": i,
        "embedding": embedding
    })
    print(f"Chunk {i}: Generated {len(embedding)}-dimensional embedding")

print(f"\nFirst 5 values of chunk 0's embedding: {chunk_embeddings[0]['embedding'][:5]}")

## Store in Neo4j

Create Document and Chunk nodes with embeddings, linked by `FROM_DOCUMENT` and `NEXT_CHUNK` relationships.

In [ ]:
def store_chunks_with_embeddings(driver, doc_path: str, chunk_data: list[dict]):
    """Store Document and Chunk nodes with embeddings."""
    with driver.session() as session:
        # Create Document
        session.run("CREATE (d:Document {path: $path})", path=doc_path)
        print(f"Created Document: {doc_path}")
        
        # Create Chunks with embeddings
        for chunk in chunk_data:
            session.run("""
                MATCH (d:Document {path: $path})
                CREATE (c:Chunk {
                    text: $text,
                    index: $index,
                    embedding: $embedding
                })
                CREATE (c)-[:FROM_DOCUMENT]->(d)
            """, path=doc_path, text=chunk["text"], 
               index=chunk["index"], embedding=chunk["embedding"])
        print(f"Created {len(chunk_data)} Chunk nodes with embeddings")
        
        # Create NEXT_CHUNK relationships
        session.run("""
            MATCH (d:Document {path: $path})<-[:FROM_DOCUMENT]-(c:Chunk)
            WITH c ORDER BY c.index
            WITH collect(c) as chunks
            UNWIND range(0, size(chunks)-2) as i
            WITH chunks[i] as c1, chunks[i+1] as c2
            CREATE (c1)-[:NEXT_CHUNK]->(c2)
        """, path=doc_path)
        print("Created NEXT_CHUNK relationships")

store_chunks_with_embeddings(driver, DOCUMENT_PATH, chunk_embeddings)

## Create Vector Index

Create a vector index for efficient similarity search (1024 dimensions for GTE).

In [ ]:
INDEX_NAME = "chunkEmbeddings"

# Drop existing index if it exists
try:
    with driver.session() as session:
        session.run(f"DROP INDEX {INDEX_NAME} IF EXISTS")
        print(f"Dropped existing index: {INDEX_NAME}")
except Exception:
    pass

# Create new vector index
create_vector_index(
    driver=driver,
    name=INDEX_NAME,
    label="Chunk",
    embedding_property="embedding",
    dimensions=1024,
    similarity_fn="cosine"
)
print(f"Created vector index: {INDEX_NAME}")

## Verify with a Test Search

In [ ]:
def vector_search(driver, embedder, query: str, top_k: int = 3):
    """Search for chunks similar to the query."""
    query_embedding = embedder.embed_query(query)
    
    with driver.session() as session:
        result = session.run("""
            CALL db.index.vector.queryNodes($index_name, $top_k, $embedding)
            YIELD node, score
            RETURN node.text as text, node.index as idx, score
            ORDER BY score DESC
        """, index_name=INDEX_NAME, top_k=top_k, embedding=query_embedding)
        return list(result)

# Test search
query = "What products does Apple make?"
print(f"Query: \"{query}\"\n")
print("=" * 60)

results = vector_search(driver, embedder, query)
for i, record in enumerate(results):
    print(f"\n[{i+1}] Score: {record['score']:.4f} (Chunk {record['idx']})")
    print(f"    {record['text']}")

## Summary

You've built the foundation for GraphRAG:

1. **Document-Chunk structure** with `FROM_DOCUMENT` and `NEXT_CHUNK` relationships
2. **Embeddings** stored on Chunk nodes using Databricks GTE model
3. **Vector index** (`chunkEmbeddings`) for fast similarity queries

---

**Next:** [GraphRAG Retrievers](02_graphrag_retrievers.ipynb) - Build complete RAG pipelines with VectorRetriever and VectorCypherRetriever.

In [ ]:
neo4j.close()